## Setup (optional dependency install)

Run the next cell **only** if you're on a fresh kernel that doesn't already have the packages. On the GPU job environment everything is preinstalled, so you can skip it.

In [ ]:
# --- Optional: install dependencies -----------------------------------------
# The GPU job env (myenv) already has all of these, so this cell is a no-op
# there and Run-All skips it. On a fresh kernel, uncomment the line below and
# run this cell once. Known-good versions: torch 2.10+cu128, albumentations 2.0.8.
# %pip install -q torch torchvision numpy opencv-python-headless albumentations matplotlib tqdm

# Train the U-Net (baseline)

Trains a small U-Net to segment the document region from the input image, then reports a **detailed test benchmark**: instance **Precision / Recall / F1 at IoU>=0.75** (+ 0.50) and pixel Dice.

Run **all cells** (one click). Uses the GPU automatically if one is available.

In [ ]:
# Standard library
import os
import sys
import random

# Third-party: arrays, image IO, the model, the input transform, plotting, progress bar
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# seg_common holds the U-Net, the dataset classes, the losses and the eval helpers, setting root path here for starting python file imports
sys.path.insert(0, "/media/beegfs/users/yashvardhan.g/HIRING_TASK/training")
import seg_common as sc

# Fix every random seed so the run is reproducible.
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

print("device:", sc.DEVICE, "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## Config

In [ ]:
# ---- training hyper-parameters ----
MODEL_NAME  = "train_no_aug.ipynb"   # checkpoint name -> checkpoints/train_no_aug.ipynb.pt
BATCH       = 16                     # images per optimisation step
EPOCHS      = 25                     # full passes over the training set
LR          = 1e-3                   # Adam learning rate
VAL_FRAC    = 0.10                   # fraction of the train set held out for validation
NUM_WORKERS = 8                      # DataLoader worker processes

# ---- where the training targets come from ----
# Masks are rasterised from the polygon column of train.csv.
TRAIN_CSV = sc.TRAIN_CSV             # /.../prepared_dataset/train.csv
sc.TRAIN_CSV = TRAIN_CSV

# This notebook only trains the model and saves a checkpoint. The test set
# (test.csv ground truth + test images) is handled separately by test.py, which
# loads the checkpoint and prints Precision / Recall / F1 (see the last cell).

print("img_size", sc.IMG_SIZE, "| batch", BATCH, "| epochs", EPOCHS)
print("train csv:", TRAIN_CSV)

## Transforms

Each image (and its mask) is resized to the working resolution and the image is normalised with the ImageNet statistics. Validation uses the same transform.

In [ ]:
# The input pipeline:
#   1. Resize both the image and its mask to IMG_SIZE x IMG_SIZE (so they stay
#      pixel-aligned).
#   2. Normalise the image with the ImageNet mean/std.
#   3. ToTensorV2 converts HWC numpy -> CHW torch tensor.
train_tf = A.Compose([
    A.Resize(sc.IMG_SIZE, sc.IMG_SIZE),
    A.Normalize(mean=sc.MEAN, std=sc.STD),
    ToTensorV2(),
])

# Validation uses exactly the same transform as training.
val_tf = train_tf

## Data — build the train/val split

Records are read from `train.csv`, shuffled with a fixed seed, and a `VAL_FRAC` slice is held out as validation. `CsvSegDataset` rasterises each record's target mask from its polygons on the fly.

In [ ]:
# Read all records, then shuffle them with a fixed seed so the split is stable.
records = sc.load_csv_records(TRAIN_CSV)
random.Random(0).shuffle(records)

# Hold out the first VAL_FRAC as validation, keep the rest for training.
n_val = int(len(records) * VAL_FRAC)
val_records = records[:n_val]
train_records = records[n_val:]

# Datasets rasterise the target mask from each record's polygons on the fly.
train_ds = sc.CsvSegDataset(train_records, train_tf)
val_ds   = sc.CsvSegDataset(val_records, val_tf)

# DataLoaders batch the samples and load them with several worker processes.
train_ld = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_ld = DataLoader(
    val_ds, batch_size=BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"train images: {len(train_ds)} | val images: {len(val_ds)}")

## Model — small U-Net (BCE + Dice loss)

In [ ]:
# The model: a small 4-level U-Net that outputs one logit per pixel.
model = sc.UNet(base=32).to(sc.DEVICE)

# Adam optimiser with a cosine-annealing schedule that decays the LR over training.
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

# Loss = BCE (per-pixel classification) + soft Dice (region overlap).
bce = nn.BCEWithLogitsLoss()

def loss_fn(logit, target):
    return bce(logit, target) + sc.dice_loss(logit, target)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params")

## Train (live progress)

In [ ]:
@torch.no_grad()
def eval_dice(loader):
    """Mean Dice over a loader: threshold the sigmoid output at 0.5, then compare
    each prediction to its ground-truth mask."""
    model.eval()
    scores = []
    for x, y in loader:
        prob = torch.sigmoid(model(x.to(sc.DEVICE)))
        pred = (prob > 0.5).cpu().numpy()
        for pred_i, y_i in zip(pred, y.numpy()):
            scores.append(sc.dice_coeff(pred_i, y_i))
    return float(np.mean(scores))


# Training loop: one pass over the data per epoch, tracking loss and val Dice.
hist = {"train_loss": [], "val_dice": []}

for ep in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_ld, desc=f"epoch {ep}/{EPOCHS}", leave=False)
    for x, y in pbar:
        x = x.to(sc.DEVICE)
        y = y.to(sc.DEVICE)

        # standard training step: forward -> loss -> backward -> update
        opt.zero_grad()
        logit = model(x)
        loss = loss_fn(logit, y)
        loss.backward()
        opt.step()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(loss=running_loss / num_batches)

    sched.step()                          # advance the LR schedule once per epoch
    val_dice = eval_dice(val_ld)          # validation Dice after this epoch

    train_loss = running_loss / num_batches
    hist["train_loss"].append(train_loss)
    hist["val_dice"].append(val_dice)
    print(f"epoch {ep:2d}  train_loss {train_loss:.4f}  val_dice {val_dice:.4f}")

## Curves

In [ ]:
# Left: training loss per epoch. Right: validation Dice per epoch.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(hist["train_loss"], marker="o")
ax[0].set_title("train loss")
ax[0].set_xlabel("epoch")

ax[1].plot(hist["val_dice"], marker="o")
ax[1].set_title("val Dice")
ax[1].set_xlabel("epoch")
ax[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Save checkpoint

In [ ]:
os.makedirs(sc.CKPT_ROOT, exist_ok=True)
ckpt = os.path.join(sc.CKPT_ROOT, f"{MODEL_NAME}.pt")
torch.save(model.state_dict(), ckpt)
print("checkpoint ->", ckpt)

## Generate your submission — `pred.csv`

Runs the trained model over every **test image** and writes `pred.csv` in the format the leaderboard expects:

| column | meaning |
|---|---|
| `image` | the test image file name (e.g. `test_00000.jpg`) |
| `polygon` | a JSON list of polygons, each a list of `[x, y]` points normalized to `[0, 1]` |

Upload the resulting `pred.csv` on the **Streamlit leaderboard page** to get your score. The evaluation labels are not distributed with this notebook — scoring happens only on the leaderboard.

In [ ]:
# Run the trained model over the TEST images and turn each predicted mask into
# normalized polygons, then write the two-column submission CSV.
import glob
import csv as _csv
import json as _json

SUBMISSION_CSV = "/media/beegfs/users/yashvardhan.g/HIRING_TASK/pred.csv"
THRESH = 0.5     # probability cutoff that turns the sigmoid output into a mask

model.eval().to(sc.DEVICE)
test_images = sorted(glob.glob(os.path.join(sc.DATA_ROOT, "test", "images", "*.jpg")))

submission_rows = []
for image_path in tqdm(test_images, desc="predicting test set"):
    image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        prob = torch.sigmoid(model(sc.preprocess(image).to(sc.DEVICE)))[0, 0].cpu().numpy()
    binary = (prob > THRESH).astype(np.uint8)
    polygons = sc.mask_to_polygons(binary)               # list of normalized polygons
    submission_rows.append((os.path.basename(image_path), _json.dumps(polygons)))

# Write CSV: header is exactly [image, polygon], one row per test image.
with open(SUBMISSION_CSV, "w", newline="") as f:
    writer = _csv.writer(f)
    writer.writerow(["image", "polygon"])
    writer.writerows(submission_rows)

print(f"wrote {len(submission_rows)} rows -> {SUBMISSION_CSV}")
print("Upload this file on the Streamlit leaderboard page to get your score.")